# 02 – Profils de comptes et features de fréquence

**Question :** comment le modèle « voit-il » qu'une transaction est anormale *pour ce compte-là* ?

Deux familles de features, calculées par les **mêmes fonctions** à l'entraînement et en temps réel (`src/ml/features.py`) :
1. **Profils de comptes** (`build_profiles`, couche *batch*) : appareils, destinataires, villes et montant moyen de chaque compte, calculés sur l'historique.
2. **Fréquence** (`add_velocity_features`) : nombre d'opérations du compte sur les 10 dernières minutes et la dernière heure, et répétitions vers le même destinataire.

In [1]:
# Préparation : on travaille depuis la racine du projet pour importer src/ et config/
import os, sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.figsize": (9, 3.6), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#E3E7EF", "axes.axisbelow": True})
AMBRE, ROUGE, TURQUOISE, GRIS = "#E89A00", "#E5384A", "#1BA597", "#C9D2E3"

In [2]:
from config.settings import DATASET_PATH, DATA_DIR
from src.ml.features import TX_SCHEMA, add_features, add_velocity_features, build_profiles
from src.ml.spark import get_spark
from pyspark.sql import functions as F

spark = get_spark("notebook-features", conf={"spark.ui.showConsoleProgress": "false"})
tx = (spark.read.csv(str(DATASET_PATH), header=True, schema=TX_SCHEMA)
      .withColumn("ts_epoch", F.to_timestamp("timestamp").cast("long")))
print(f"{tx.count():,} transactions chargées dans Spark".replace(",", " "))

203 141 transactions chargées dans Spark


## 1. Un cas réel : une prise de contrôle de compte
On prend une vraie fraude de type « prise de contrôle » dans les données, et on regarde le compte victime **avant** la fraude.

In [3]:
sc = pd.read_csv(DATA_DIR / "transactions_scenarios.csv", usecols=["transaction_id", "fraud_scenario"])
ato_ids = sc.loc[sc.fraud_scenario == "ATO", "transaction_id"].tolist()
fraude = (tx.filter(F.col("transaction_id").isin(ato_ids[:400])).filter("tx_type = 'TRANSFER'")
          .orderBy(F.desc("amount")).first())
victime, t_fraude = fraude["sender_id"], fraude["ts_epoch"]
print(f"Compte {victime} : transfert de {fraude['amount']:,.0f} XAF vers {fraude['receiver_id']}, "
      f"le {fraude['timestamp'][:16]}, appareil {fraude['device_id']}".replace(",", " "))

Compte 237600011902 : transfert de 415 000 XAF vers 237600010372  le 2026-09-03T09:12  appareil dev-9a08fbd710dc


In [4]:
# Profil du compte calculé UNIQUEMENT sur ce qui précède la fraude (comme en production)
historique = tx.filter(F.col("ts_epoch") < t_fraude)
profil = build_profiles(historique).filter(F.col("p_sender_id") == victime).first()
print(f"Opérations passées : {profil['p_n_tx']}")
print(f"Montant moyen      : {profil['p_avg_amount']:,.0f} XAF (maximum {profil['p_max_amount']:,.0f})".replace(",", " "))
print(f"Appareils connus   : {sorted(profil['p_devices'])}")
print(f"Villes habituelles : {sorted(profil['p_cities'])}")
print(f"Destinataires déjà payés : {len(profil['p_receivers'])}")

Opérations passées : 2
Montant moyen      : 5 250 XAF (maximum 6 500)
Appareils connus   : ['dev-5f86fb04256e']
Villes habituelles : ['Douala']
Destinataires déjà payés : 2


In [5]:
# Les features que reçoit le modèle pour cette transaction
profils = build_profiles(historique)
ligne = add_features(add_velocity_features(tx.filter(F.col("ts_epoch") <= t_fraude))
                     .filter(F.col("transaction_id") == fraude["transaction_id"]), profils)
cols = ["amount", "amount_vs_avg", "device_is_new", "receiver_is_new", "city_is_new", "is_night",
        "sender_emptied", "sender_n_10min", "pair_n_10min"]
ligne.select(*cols).toPandas().T.rename(columns={0: "valeur"})

,valeur
amount,415000.000000
amount_vs_avg,79.047619
device_is_new,1.000000
receiver_is_new,1.000000
city_is_new,1.000000
is_night,0.000000
sender_emptied,0.000000
sender_n_10min,1.000000
pair_n_10min,1.000000


Lecture : **appareil jamais vu**, **destinataire jamais payé**, montant plusieurs fois supérieur à la moyenne du compte.
Pris isolément, un montant de cette taille n'a rien d'anormal ; c'est l'**écart aux habitudes** qui trahit la fraude.

## 2. Un fractionnement par un agent : les features de fréquence
Un même client fait plusieurs petits dépôts chez **le même agent**, à quelques dizaines de secondes d'écart.
Chaque dépôt est banal ; c'est la **répétition** qui est suspecte.

In [6]:
split_ids = sc.loc[sc.fraud_scenario == "AGENT_SPLIT", "transaction_id"].tolist()
une = tx.filter(F.col("transaction_id") == split_ids[0]).first()
serie = (add_velocity_features(tx.filter((F.col("sender_id") == une["sender_id"])
                                         & (F.abs(F.col("ts_epoch") - une["ts_epoch"]) < 1800)))
         .orderBy("ts_epoch")
         .select(F.substring("timestamp", 12, 8).alias("heure"), "tx_type", "receiver_id", "amount",
                 "pair_n_10min", "sender_n_10min"))
serie.toPandas()

,heure,tx_type,receiver_id,amount,pair_n_10min,sender_n_10min
0,08:56:40,CASH_IN,237700000026,4000.0,1.0,1.0
1,08:58:16,CASH_IN,237700000026,17000.0,2.0,2.0
2,08:58:49,CASH_IN,237700000026,9500.0,3.0,3.0
3,08:59:36,CASH_IN,237700000026,16000.0,4.0,4.0
4,09:01:07,CASH_IN,237700000026,19000.0,5.0,5.0


La colonne `pair_n_10min` (opérations vers le même destinataire dans les 10 dernières minutes) **monte à chaque dépôt** : c'est ce signal qui permet de détecter le fractionnement.

## 3. Même calcul à l'entraînement et en temps réel
En temps réel, Spark reçoit les transactions par micro-lots de 5 s. Le scoring garde **la dernière heure** de transactions en mémoire, y ajoute le nouveau lot, et applique **exactement la même fonction** `add_velocity_features`.
Le test `tests/test_velocity_features.py` vérifie que ce calcul par lots donne les **mêmes valeurs** qu'un calcul d'un seul coup : il n'y a donc pas de décalage entre le modèle entraîné et le modèle servi.

In [7]:
spark.stop()